# 04 · Self-distillation: train agents on the base model's best answers

Runtime > Change runtime type > **T4 GPU**.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
!nvidia-smi

In [ ]:
import getpass, os
GITHUB_USER = 'natitedros'
REPO = 'CMVA-context-based-aggregation'
token = getpass.getpass('GitHub token (leave blank if repo is public): ')
auth = f'{token}@' if token else ''
%cd /content
!rm -rf /content/repo
!git clone -q https://{auth}github.com/{GITHUB_USER}/{REPO}.git /content/repo
%cd /content/repo
del token, auth

In [ ]:
!pip install -q -r requirements.txt
!pip uninstall -y -q torchao

In [ ]:
import os
os.environ['PROJECT_ROOT'] = '/content/drive/MyDrive/NLP-Final'
os.environ['SCRATCH_DIR'] = '/content/scratch'
DISTILLED = os.environ['PROJECT_ROOT'] + '/data/agent_data_distilled'

### 1. Smoke test: 32 questions
Checks the whole pipeline in a few minutes. Delete its outputs afterwards (next cell).

In [ ]:
!python src/distill_data.py --n_questions 32 --chunk 16

In [ ]:
# Remove the smoke-test outputs before the real run
!rm -rf "$PROJECT_ROOT/data/distill" "$PROJECT_ROOT/data/agent_data_distilled"

### 2. Real run: 2,000 questions × 5 answers
Resumable: if Colab disconnects, rerun the setup cells and this cell, and it continues from the last saved chunk.

In [ ]:
!python src/distill_data.py

### 3. The gap check
`gap_vs_random` should be clearly positive for every value, and `mean_selected` for care should be well above 0.87.

In [ ]:
import json
print(json.dumps(json.load(open(DISTILLED + '/stats.json')), indent=2))

### 4. Train the care agent in both arms, plus the control

In [ ]:
!python src/train_agent.py --mode lora --value care --data_dir "$PROJECT_ROOT/data/agent_data_distilled" --tag _sd
!python src/train_agent.py --mode full --value care --data_dir "$PROJECT_ROOT/data/agent_data_distilled" --tag _sd
!python src/train_agent.py --mode lora --value control --data_dir "$PROJECT_ROOT/data/agent_data_distilled" --tag _sd

### 5. Score all three on the 500 held-out questions

In [ ]:
A = os.environ['PROJECT_ROOT'] + '/models/agents'
!python src/generate_and_score.py --tag lora_care_sd --adapter "{A}/lora_care_sd" --n 500
!python src/generate_and_score.py --tag full_care_sd --model "{A}/full_care_sd" --n 500
!python src/generate_and_score.py --tag lora_control_sd --adapter "{A}/lora_control_sd" --n 500

### 6. Compare with the base model

In [ ]:
import pandas as pd
R = os.environ['PROJECT_ROOT'] + '/results'
VALUES = ['care', 'fairness', 'loyalty', 'authority', 'sanctity']
tags = ['pt', 'full_care_sd', 'lora_care_sd', 'lora_control_sd']
table = pd.DataFrame({t: {v: json.load(open(f'{R}/alignment_{t}.json'))[v]['expected_reward'] for v in VALUES} for t in tags})
table